### 梯度下降带Armijo线搜索

使用固定$\alpha$的前提，要求函数的梯度满足Lipschitz条件。且注意到，即使函数光滑，这个给出的L是全局的，而非局部的，可能导致取值保守。

在机器学习常见的一个操作，加L1正则化；即使损失函数是光滑，加L1正则（绝对值函数部分），整体就是非光滑函数。这时就需要线搜索确定$\alpha$.

线搜索先确定下降方向$p_k$，再沿该方向选择$\alpha_k$.



#### 精确线搜索

精确线搜索求解一维问题：

$$\alpha_k=\mathop{arg min_{\alpha >0 }} f(x_k+\alpha p_k) $$

除了特殊二次结构外，精确求解通常不划算。多次高精度计算一维最优步长，未必能减少总体计算量。

#### Armijo 条件

Armijo 条件要求

$$  f(x_k+\alpha_k p_k)\le f(x_k)+c_1 \alpha_k \nabla f(x_k)^T p_k, \quad 0<c_1<1 $$

右端是线性预测下降量的一部分。常用$c_1=10^{-4}$

回溯搜索从 $\alpha=1 $开始，若条件不成立，就令$\alpha \gets \rho \alpha$, 其中 $\rho \in (0,1)$ ,常取0.5。只要方向满足：

$$ \nabla f(x_k)^T p_k<0$$

函数连续可微，回溯最终会接受一个正步长。

#### 引入实际例子

在一台工业离心泵 布置30个传感器/测点，（比如用来监测 震动、温度、压力、流量、电流等状态量）

数据文件`equipment_failure_risk`的`sensor_01`至`sensor_30`作为通用字段名（避免模拟物理单位），采用10分中记录一次全部30个测点，形成8000条数据

数据跨8个周期，每个周期显式包含"运行-故障" 状态，从上一个周期的故障到下一个周期的运行，隐含维修复位。

训练数据 使用1-6周期，测试数据用7-8周期。

数据标准化：
$$ \bar{x}_{ij}=\frac{x_{ij}-\bar{x}_{j,train}}{s_{j,train}}$$

#### 正则化逻辑回归

对于样本 $(x_i,y_i)$,其中 $y_i \in \{0,1\}$, 定义 

$$ z_i=x_i^T w, \quad \sigma(z_i)=\frac{1}{1+exp(-z_i)}$$

逻辑回归是判别式模型，依据y=0,y=1的边界判断分类；
先将$boldsymbol{x}_i$ 映射到实值$z_i$,在一致信息，计算将$y_i$判别1或0；由此利用似然函数，让每个样本属于真实分类标记的判别概率越大越好（即判断准确）。

不妨令 $p(y_i=1|x_i)=\frac{1}{1+e^{-z}}$

$$\begin{aligned}p_i &=y_ip(y_i=1|x_i)+(1-y_i)p(y_i=0|x_i)\\
&=\frac{y_i+(1-y_i)e^{-z_i}}{1+e^{-z_i}}\\
\end{aligned}$$

分子分母同乘$e^{z_i}$，得到$\frac{y_ie^{z_i}+(1-y_i)}{e^{z_i}+1}$


全体样本概率相乘得似然函数，取对数方便后续计算
$$L(w)=\sum_{i=1}^n \log(y_ie^{z_i}+1-y_i)-\log(1+e^{z_i})$$


$y_i$是二元变量,带入第一项可简化为$y_i z_i$

求平均并加入L2正则项

$$J(w)=\frac{1}{n} \sum_{i=1}^n(y_iz_i -\log(1+e^{z_i}))-\frac{\lambda}{2}||\tilde{w}||_2^2$$
$\tilde{w}$不包含截距项。


梯度下降用于求最小，所以加入负号：

$$F(w)=\frac{1}{n} \sum_{i=1}^n(\log(1+e^{z_i})- y_iz_i)+\frac{\lambda}{2}||\tilde{w}||_2^2$$

其中Softplus=$\log (1+e^{z})$  计算时，$z$ 增大，$e^z$上溢出。

改写为 $\log (1+e^{z})=max(z,0)+\log(1+e^{-|z|})$

#### 梯度更新

梯度 $$\begin{aligned} \nabla f(w) &=\frac{\partial{F(w)}}{\partial{w}}\\
&=\frac{1}{n}\sum_{i=1}^{n} (\frac{e^{z_i}x_i}{1+e^{z_i}}-y_ix_i)+\lambda R w
\end{aligned}$$

其中$R=diag(0,1,...,1)$, 不含截距项

不看后面正则项，前面部分恰为 特征$\times$（预测值-真实值）.

In [18]:
import importlib
import model
importlib.reload(model)

<module 'model' from 'd:\\vsCode_workspace\\gradient_descent\\model.py'>

In [19]:

import numpy as np
from time import perf_counter

In [20]:
# 返回 当前步 要使用的 alpha, 以及 线搜索迭代数
def armijo(weights,direction,features,labels,initial_alpha=1.0,c1=1e-4,shrink=0.5):
    # 条件不成立  \alpha 收缩
    value=model.objective(weights,features,labels)
    grad=model.gradient(weights,features,labels)

    # \nabla f(x_k)^T p_k<0  p_k下降方向
    slope=np.dot(grad,direction)
    if slope>=0.0:
        raise ValueError("Armijo line search requires a descent direction")
    alpha=initial_alpha
    iters=0

    # f(x_{k+1})<=f(x_k)+c_1 \alpha_k \nabla f(x_k)^Tp_k
    while model.objective(weights+alpha*direction,features,labels)>value+c1*alpha*slope:
        alpha*=shrink
        iters+=1
        if alpha<1e-14:
            raise RuntimeError("Armijo line search failed: alpha is too small")
    return alpha, iters+1

In [21]:
def gradient_descent(features,labels,max_iter=500,tolerance=1e-7):
    weights=np.zeros(features.shape[1])
    start=perf_counter()
    for iter in range(1,max_iter+1):
        grad=model.gradient(weights,features,labels)
        if np.linalg.norm(grad)<=tolerance:
            break
        direction=-grad
        alpha,local_steps=armijo(weights,direction,features,labels)
        weights+=alpha*direction
    return model.make_record("Armijo-GD",weights,features,labels,iter,perf_counter()-start)

In [22]:
def test_gd_arm():
    x_train, y_train, x_test, y_test, metadata = model.load_data()
    results=gradient_descent(x_train,y_train)
    accuracy, precision,recall, log_loss = model.classification_metrics(
            results["weights"], x_test, y_test)
    print(
        f"method: {results.get('method')}\n"
        f"accuracy: {accuracy:.4f}\n"
        f"precision: {precision:.4f}\n"
        f"recall: {recall:.4f}\n"
        f"log_loss: {log_loss:.4f}"
    )

test_gd_arm()


method: Armijo-GD
accuracy: 0.9781
precision: 0.9586
recall: 0.8968
log_loss: 0.0619
